Fitting with user-defined functions
===

There are numerous ways to express functions in ROOT (https://root.cern.ch/doc/master/classTF1.html) in this example we'll explore a general method of defining fit functions.  Again we will use the ROOT fitting framework for the next couple of exercies to illustrate some general featues of the non-linear fitting problems.  Some of the examples we see may be replicated in scipy.optimize, but not neceassarily all of them.

ROOT classes used here:
* [TF1](https://root.cern.ch/doc/master/classTF1.html): 1D function
* [TH1F](https://root.cern.ch/doc/master/classTH1.html): 1D histogram (the content of each bin is described by a float)

In [ ]:
import ROOT as r
%jsroot off

ERROR in cling::CIFactory::createCI(): cannot extract standard library include paths!
Invoking:
  LC_ALL=C x86_64-conda-linux-gnu-c++  -O3 -DNDEBUG -xc++ -E -v /dev/null 2>&1 | sed -n -e '/^.include/,${' -e '/^ \/.*++/p' -e '}'
Results was:
With exit code 0
C system headers (glibc/Xcode/Windows SDK) must be installed.
/sfs/ceph/standard/phys56xx/miniforge3/envs/phys56xx/etc/cling/std.modulemap:69:12: error: header 'compare' not found
    header "compare"
           ^
input_line_1:1:10: note: submodule of top-level module 'std' implicitly imported here
#include <new>
         ^
Warning in cling::IncrementalParser::CheckABICompatibility():
  Failed to extract C++ standard library version.
In file included from input_line_4:36:
In file included from /../lib/gcc/x86_64-redhat-linux/8/../../../../include/c++/8/cassert:43:
/../lib/gcc/x86_64-redhat-linux/8/../../../../include/c++/8/x86_64-redhat-linux/bits/c++config.h:3:10: fatal error: 'bits/wordsize.h' file not found
#include <bits/wordsiz

First we define a function and store it in a TF1 object.  In this example we will use Pythonic interfaces, next time we'll see how the same patterns can be used in C++.

In [ ]:
from math import pow, exp

# A function producing two peaks on top of an exponentialy 
# falling background.  Depends on several parameters.
# Generic interface for fcn of n input-values and m parameters
# Functions with this interface may be used to construct a "TFunction" or TF1
# vx is the independent value(s): array like
# p is the parameter value(s): array like

def myfunction(vx, par):
    x=vx[0]
    # background model
    bkgScale=par[0]
    alpha=par[1]
    beta=par[2]
    background = pow(x/beta,-1.0*alpha)
    # first peak, Gaussian model
    A1=par[3]
    mu1=par[4]
    sig1=par[5]
    peak1=A1*exp(-0.5*(x-mu1)*(x-mu1)/sig1/sig1)
    # second peak, Gaussian model
    A2=par[6]
    mu2=par[7]
    sig2=par[8]
    peak2=A2*exp(-0.5*(x-mu2)*(x-mu2)/sig2/sig2)
    return bkgScale*background+peak1+peak2

xrange=(300,1000)
f1 = r.TF1("f1",myfunction,xrange[0],xrange[1],9)  # xrange 300<=x<=1000, there are 9 parameters in this function
f1.SetNpx(500)  # use large number of points in drawing function to resolve small details better
f1.SetParameters(1e9,4.7,40,5000,490,2,1200,800,25)  # define the parameters
f1.SetParNames("BkgScale","alpha","beta","A1","mu1","sig1","A2","mu2","sig2")  # optional, but nice

In [ ]:
tc=r.TCanvas()
f1.Draw()
tc.Draw()

Generate random data according to this distribution

In [ ]:
entries=100000
ranHist1 = r.TH1F("ranHist1", "Random Histogram for my PDF;x;entries",500,xrange[0],xrange[1]);
ranHist1.FillRandom("f1",entries)

In [ ]:
ranHist1.Draw("e")
tc.Draw()

 Now "pretend" that we don't know the paramaters used to generate the data.
 
 All fits begin with initial guesses at the best parameter values

In [ ]:
f1.SetParameters(1e6,4,80,200,550,3,200,700,20)
ranHist1.Draw("e")
f1.Draw("same")
tc.Draw()

To get better qualitative agreement try:

In [ ]:
f1.SetParameters(0.5e6,4,80,200,550,3,200,700,20)
ranHist1.Draw("e")
f1.Draw("same")
tc.Draw()

Try to fit the function to the data:

In [ ]:
result=ranHist1.Fit(f1,"E")
f1=ranHist1.GetFunction("f1")

Notice that there is a problem here. <br>
Look at the result:

In [ ]:
print(f'chi^2: {f1.GetChisquare()}, nDOF: {f1.GetNDF()}, p-value: {f1.GetProb()}')
ranHist1.Draw("e")
f1.Draw("same")
tc.Draw()

Did the fit work? Did it find both peaks? Probably not. In general, you can't count on complex fits to converge without carefully adjusting the starting parameters. Sure, in this example, you could peek at the parameters used to generate the data, but that's not an option in the real world. Go through the process of adjusting the parameters and replotting the function to get a better representation of the data. Then try a fit and see if you can extract the parameters describing the peeks.

In [ ]:
# extract the parameters and their (parabolic) errors
popt = []
perr = []
for i in range(f1.GetNpar()):
    popt.append(f1.GetParameter(i))
    perr.append(f1.GetParError(i))
    print(f'f1.GetParName(i): {popt[i]:10.2f} +- {perr[i]:10.2f}')

For you to try
===
In the file datadist.root you will find a histogram representing data from an unknown distribution.

* Develop your own fitting function/model and see how well you can fit this distribution. 
* You may need to try a variety of functions.
* Include a plot of your best fit at the bottom of this notebook.
* Include your p-value for the best fit and describe how you settled on this fit versus others.
* Show your best fit parameters and their errors
* Plot the fit residuals, eg for each bin plot (fit-data)/data_uncertianty.  For a good fit the points should randomy fluctuate around 0 (eg no large, contiguous regions above or below 0)
* Plot the pull distribution (for a good fit this should be consisten with a normal distribution w/ $\mu=0,\sigma=1$

For this notebook it is assumed that you'll work with ROOT.

In [ ]:
tf=r.TFile("datadist.root")
hist=tf.Get("h")
tc=r.TCanvas()
hist.Draw()
tc.Draw()

In [ ]:
# your work here
from math import sin, sqrt

def quad(vx, par):
    x=vx[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return A*x**2 + B*x + C

def gaus(vx, par):
    x=vx[0]
    A = par[0]
    mu = par[1]
    sig = par[2]
    back = par[3]
    return A*exp(-0.5*(x-mu)*(x-mu)/sig/sig)+back

def sinfit(vx, par):
    x=vx[0]
    A = par[0]
    k = par[1]
    back = par[2]
    return A*sin(k*x)+back

xrange=(0,12)

quadratic = r.TF1("quadratic",quad,xrange[0],xrange[1],3)  # xrange 0<=x<=12, there are 3 parameters in this function
quadratic.SetNpx(500)  # use large number of points in drawing function to resolve small details better
quadratic.SetParameters(-1,16,35)  # define the parameters
quadratic.SetParNames("A", "B", "C")  # optional, but nice

gaussian = r.TF1("gaussian",gaus,xrange[0],xrange[1],4)  # xrange 0<=x<=12, there are 3 parameters in this function
gaussian.SetNpx(500)  # use large number of points in drawing function to resolve small details better
gaussian.SetParameters(105,8,6,-5)  # define the parameters
gaussian.SetParNames("A", "mu", "sigma", "background")  # optional, but nice

sinusoid = r.TF1("sinusoid",sinfit,xrange[0],xrange[1],3)  # xrange 0<=x<=12, there are 3 parameters in this function
sinusoid.SetNpx(500)  # use large number of points in drawing function to resolve small details better
sinusoid.SetParameters(65,0.2,0,+35)  # define the parameters
sinusoid.SetParNames("A", "k", "background")  # optional, but nice

In [ ]:
result_quad=hist.Fit(gaudratic,"E")
f1=hist.GetFunction("quadratic")

result_gaus=hist.Fit(gaussian,"E")
f2=hist.GetFunction("gaussian")

result_sin=hist.Fit(sinusoid,"E")
f3=hist.GetFunction("sinusoid")

print(f'chi^2: {f1.GetChisquare()}, nDOF: {f1.GetNDF()}, p-value: {f1.GetProb()}')
print(f'chi^2: {f2.GetChisquare()}, nDOF: {f2.GetNDF()}, p-value: {f2.GetProb()}')
print(f'chi^2: {f3.GetChisquare()}, nDOF: {f3.GetNDF()}, p-value: {f3.GetProb()}')

The fit with the best chi^2 is what I will use for my best fit.

In [ ]:
tc=r.TCanvas()
hist.Draw()
f_.Draw("same")
tc.Draw()
print(f'p-value for best fit: {f_.GetProb()}\n')
print('Fit Parameters and Errors')
print(f'_:{f_.GetParameter(0)}' +- {f_.GetParError(0)})
print(f'_:{f_.GetParameter(1)}' +- {f_.GetParError(1)})
print(f'_:{f_.GetParameter(2)}' +- {f_.GetParError(2)})

In [ ]:
bins = []
res = []
for i in range(1, hist.GetNbinsX()+1):
    bins.append(i)
    xi = hist.GetBinCenter(i)
    yfit = f_.Eval(xi)
    yi = hist.GetBinContent(i)
    res.append((yfit-yi)/sqrt(yi))

scatter = r.TScatter(len(bins), bins, res)
scatter.SetTitle("Residuals;bin number;residual value")
res_hist = r.TH1F("res_hist","res_hist",100,-2,2)
for i in res:
    res_hist.Fill(i)

tc2 = r.TCanvas()
tc2.Divide(2,1)
tc2.cd(1)
scatter.Draw()
tc2.cd(2)
res_hist.Draw()
tc2.Draw()